# Dia 5 — Orquestrador e sub-agente de skills

Hoje chegamos ao conceito mais avançado do curso: **agentes que delegam para outros agentes**.

O problema que resolvemos:
> O usuário pede genericamente *"envia um alerta de falha"*.  
> O sistema precisa saber **como** escrever esse alerta — tom, formato, campos obrigatórios.  
> Esse conhecimento fica em arquivos `.md` chamados **skills**.

### Arquitetura

```
Usuário
   ↓ pedido genérico
Agente Orquestrador
   ↓ delega leitura de skills
Sub-agente de Skills
   ↓ read_skill(header_only=True)  → lê cabeçalhos de todos os .md
   ↓ decide qual skill é relevante
   ↓ read_skill(header_only=False) → lê skill completa
   ↓ retorna contexto + instruções
Agente Orquestrador
   ↓ usa send_email com as instruções recebidas
   ↓ envia para todos os usuários
```

**Separação de responsabilidades:**
- Sub-agente: só lê e decide. Nunca envia e-mail.
- Orquestrador: só envia. Nunca lê skills diretamente.

| Parte | Tema |
|---|---|
| Setup | Conexões, credenciais, skills em disco |
| A | Tool `read_skill` com suporte a `header_only` |
| B | Sub-agente de skills |
| C | Agente orquestrador que delega e envia |
| D | Testes end-to-end |

---

## Setup

In [ ]:
# Célula 1 — Instalar dependências
!pip install -q langchain-anthropic langchain langgraph

In [ ]:
# Célula 2 — Credenciais
PROXY_URL      = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN    = "xpto_aluno-01"

EMAIL_URL      = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN    = "aluno-01"
EMAIL_PASSWORD = "1234"

# Pasta onde as skills estão salvas
# O professor disponibiliza esta pasta antes da aula
SKILLS_FOLDER  = "/content/skills"  # ajuste se necessário

print("Credenciais carregadas.")

In [ ]:
# Célula 3 — Criar a pasta de skills e os arquivos .md
# (Em produção o professor entregaria esses arquivos prontos;
#  aqui criamos programaticamente para o notebook ser autossuficiente)

import os

os.makedirs(SKILLS_FOLDER, exist_ok=True)

SKILL_FALHA = """# skill: alerta_falha
# descricao: Instrui como redigir um e-mail de alerta de falha crítica no sistema.
# palavras-chave: falha, erro, crítico, down, indisponível

---

## Instruções para o agente orquestrador

Você deve redigir um e-mail de **alerta de falha crítica** seguindo rigorosamente o formato abaixo.

### Formato obrigatório do corpo do e-mail

🚨 ALERTA DE FALHA CRÍTICA

**Sistema afetado:** [nome do sistema ou componente]
**Severidade:** CRÍTICA
**Horário da ocorrência:** [horário informado ou "não informado"]

**O que aconteceu:**
[descrição breve da falha em 1-2 frases]

**Impacto:**
[descreva quem ou o que é afetado]

**Ação necessária:**
[o que o time deve fazer agora]

— Sistema de Monitoramento Automático

### Regras de formatação
- Sempre iniciar com o emoji 🚨 e o título em CAIXA ALTA
- O campo Severidade deve ser sempre CRÍTICA
- Assunto do e-mail deve seguir o padrão: [FALHA] <nome do sistema>
- Destinatários: todos os usuários da lista fornecida pelo orquestrador
"""

SKILL_PRODUCAO = """# skill: alerta_producao
# descricao: Instrui como redigir um e-mail comemorativo de meta de produção atingida.
# palavras-chave: produção, meta, atingida, sucesso, meta batida, resultado

---

## Instruções para o agente orquestrador

Você deve redigir um e-mail de **celebração de meta de produção atingida** seguindo o formato abaixo.

### Formato obrigatório do corpo do e-mail

✅ META DE PRODUÇÃO ATINGIDA

**Linha / Área:** [nome da linha ou área]
**Meta estabelecida:** [valor da meta]
**Resultado alcançado:** [valor atingido]
**Período:** [período informado ou "turno atual"]

Parabéns ao time! 🎉
[mensagem motivacional curta de 1-2 frases]

**Próximos passos:**
[o que acontece agora]

— Sistema de Monitoramento de Produção

### Regras de formatação
- Sempre iniciar com o emoji ✅ e o título em CAIXA ALTA
- O campo Resultado alcançado deve aparecer em destaque
- Assunto do e-mail deve seguir o padrão: [PRODUÇÃO] Meta atingida — <área>
- Destinatários: todos os usuários da lista fornecida pelo orquestrador
"""

with open(f"{SKILLS_FOLDER}/alerta_falha.md", "w") as f:
    f.write(SKILL_FALHA)

with open(f"{SKILLS_FOLDER}/alerta_producao.md", "w") as f:
    f.write(SKILL_PRODUCAO)

arquivos = os.listdir(SKILLS_FOLDER)
print(f"Skills disponíveis em '{SKILLS_FOLDER}':")
for a in arquivos:
    print(f"  - {a}")

In [ ]:
# Célula 4 — LLM e login no servidor de e-mail
import requests
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=1024,
)

resp = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print("LLM:", resp.content)

login = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN, "password": EMAIL_PASSWORD,
})
assert login.status_code == 200
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Email server: OK →", login.json())

In [ ]:
# Célula 5 — Tool send_email (mesma dos dias anteriores)
from langchain_core.tools import tool
from typing import Optional

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: E-mails em cópia, separados por vírgula (opcional)
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc
    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

print("Tool send_email pronta.")

---
## Parte A — Tool `read_skill`

Esta é a tool exclusiva do sub-agente.  
Ela aceita dois modos de leitura:

- `header_only=True` → lê apenas as primeiras linhas de **todos** os arquivos da pasta  
  (rápido, barato, serve para o sub-agente decidir qual skill usar)
- `header_only=False` + `filename` → lê o conteúdo completo de um arquivo específico  
  (retorna as instruções completas para o orquestrador)

Essa separação imita um padrão real de sistemas de produção:  
*scan rápido → decidir → ler completo*.

In [ ]:
# Célula 6 — Tool: read_skill

HEADER_LINES = 4  # quantas linhas considerar como cabeçalho

@tool
def read_skill(header_only: bool, filename: Optional[str] = None) -> str:
    """Lê arquivos de skill (.md) da pasta de skills.

    Modo 1 — Escanear cabeçalhos (use primeiro para decidir qual skill usar):
        header_only=True, filename=None
        Retorna as primeiras linhas de TODOS os arquivos .md da pasta.

    Modo 2 — Ler skill completa (use após decidir qual arquivo é relevante):
        header_only=False, filename='nome_do_arquivo.md'
        Retorna o conteúdo completo do arquivo especificado.

    Args:
        header_only: True para escanear cabeçalhos, False para ler completo.
        filename: Nome do arquivo .md (necessário quando header_only=False).
    """
    if header_only:
        # Modo 1: retorna cabeçalhos de todos os arquivos
        resultado = []
        try:
            arquivos = [f for f in os.listdir(SKILLS_FOLDER) if f.endswith(".md")]
        except FileNotFoundError:
            return f"Pasta de skills não encontrada: '{SKILLS_FOLDER}'"

        if not arquivos:
            return "Nenhuma skill encontrada na pasta."

        for arq in sorted(arquivos):
            caminho = os.path.join(SKILLS_FOLDER, arq)
            with open(caminho, "r") as f:
                linhas = [f.readline() for _ in range(HEADER_LINES)]
            resultado.append(f"=== {arq} ===")
            resultado.append("".join(linhas).strip())
            resultado.append("")

        return "\n".join(resultado)

    else:
        # Modo 2: lê o arquivo completo
        if not filename:
            return "Erro: filename é obrigatório quando header_only=False."

        caminho = os.path.join(SKILLS_FOLDER, filename)
        if not os.path.exists(caminho):
            return f"Arquivo '{filename}' não encontrado em '{SKILLS_FOLDER}'."

        with open(caminho, "r") as f:
            return f.read()


print("Tool read_skill pronta.")

In [ ]:
# Célula 7 — Testar read_skill diretamente (sem agente)

print("=== Modo 1: escanear cabeçalhos ===")
print(read_skill.invoke({"header_only": True}))

print("\n=== Modo 2: ler skill completa ===")
print(read_skill.invoke({"header_only": False, "filename": "alerta_falha.md"}))

---
## Parte B — Sub-agente de skills

O sub-agente tem acesso **apenas** à tool `read_skill`.  
Ele não sabe enviar e-mails — só lê e decide.

Seu fluxo interno:
1. `read_skill(header_only=True)` → escaneia os cabeçalhos
2. Decide qual arquivo é mais relevante para o pedido
3. `read_skill(header_only=False, filename=...)` → lê a skill completa
4. Retorna as instruções para o orquestrador

In [ ]:
# Célula 8 — Sub-agente de skills
from langchain.agents import create_agent

sub_agente_skills = create_agent(
    model=llm,
    tools=[read_skill],
    system_prompt=(
        "Você é um sub-agente especializado em skills de comunicação.\n"
        "Seu único trabalho é encontrar e retornar as instruções corretas de formatação.\n\n"
        "Sempre siga este processo:\n"
        "1. Use read_skill(header_only=True) para escanear os cabeçalhos de todas as skills.\n"
        "2. Com base no pedido e nos cabeçalhos, identifique qual arquivo .md é mais relevante.\n"
        "3. Use read_skill(header_only=False, filename=<arquivo>) para ler a skill completa.\n"
        "4. Retorne o conteúdo completo da skill sem modificações — o orquestrador usará essas "
        "instruções para redigir e enviar os e-mails.\n\n"
        "Você NÃO envia e-mails. Apenas retorna instruções."
    ),
)

print("Sub-agente de skills criado.")

In [ ]:
# Célula 9 — Testar o sub-agente isoladamente

def invocar_sub_agente(pedido: str) -> str:
    """Invoca o sub-agente e retorna as instruções de skill."""
    resultado = sub_agente_skills.invoke(
        {"messages": [{"role": "user", "content": pedido}]},
        config={"recursion_limit": 10},
    )
    return resultado["messages"][-1].content


print("=== Sub-agente: pedido de alerta de falha ===")
instrucoes = invocar_sub_agente("preciso enviar um alerta de falha no sistema")
print(instrucoes[:600], "...")

In [ ]:
# Célula 10 — Modo verbose: ver o sub-agente em ação (read → decide → read)

def invocar_sub_agente_verbose(pedido: str):
    resultado = sub_agente_skills.invoke(
        {"messages": [{"role": "user", "content": pedido}]},
        config={"recursion_limit": 10},
    )
    for msg in resultado["messages"]:
        tipo = msg.__class__.__name__
        print(f"\n{'='*40}")
        print(f"  {tipo}")
        print(f"{'='*40}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → tool: {tc['name']}")
                print(f"     args: {tc['args']}")
        elif hasattr(msg, "content"):
            conteudo = str(msg.content)
            print(conteudo[:400] + ("..." if len(conteudo) > 400 else ""))

invocar_sub_agente_verbose("preciso do alerta de produção atingida")

---
## Parte C — Agente orquestrador

O orquestrador recebe o pedido do usuário e coordena tudo:

1. Chama o sub-agente para obter as instruções de skill
2. Usa `send_email` para enviar para todos os destinatários

Ele tem acesso a `send_email` mas **não** a `read_skill` — a leitura de skills é responsabilidade exclusiva do sub-agente.

A chamada ao sub-agente é feita via uma **tool wrapper** — assim o orquestrador pode invocá-lo da mesma forma que invoca qualquer outra tool.

In [ ]:
# Célula 11 — Wrapper do sub-agente como tool do orquestrador

# Lista de todos os usuários que receberão os alertas
TODOS_USUARIOS = [
    "aluno01@curso.ia",
    "aluno02@curso.ia",
    "aluno03@curso.ia",
    "aluno04@curso.ia",
    "aluno05@curso.ia",
]

@tool
def consultar_skill(tipo_de_alerta: str) -> str:
    """Consulta o sub-agente de skills para obter as instruções de formatação de um alerta.

    Use esta tool quando precisar saber COMO escrever um e-mail de alerta.
    Retorna as instruções completas de formatação e o padrão de assunto a usar.

    Args:
        tipo_de_alerta: Descrição do tipo de alerta (ex: 'falha crítica', 'meta de produção atingida')
    """
    print(f"  [Orquestrador → Sub-agente]: consultando skill para '{tipo_de_alerta}'")
    instrucoes = invocar_sub_agente(tipo_de_alerta)
    return instrucoes


@tool
def listar_destinatarios() -> str:
    """Retorna a lista de todos os usuários que devem receber alertas do sistema.

    Use esta tool quando o usuário pedir para enviar para 'todos' ou 'todos os usuários'.
    """
    return "Destinatários:\n" + "\n".join(f"- {u}" for u in TODOS_USUARIOS)


print("Tools do orquestrador:", [consultar_skill.name, listar_destinatarios.name, send_email.name])

In [ ]:
# Célula 12 — Agente orquestrador

orquestrador = create_agent(
    model=llm,
    tools=[consultar_skill, listar_destinatarios, send_email],
    system_prompt=(
        "Você é o agente orquestrador de alertas.\n"
        "Quando o usuário pedir para enviar um alerta, siga sempre este processo:\n\n"
        "1. Use consultar_skill para obter as instruções de formatação do tipo de alerta.\n"
        "2. Use listar_destinatarios para obter a lista completa de usuários.\n"
        "3. Redija o e-mail seguindo RIGOROSAMENTE as instruções retornadas pela skill.\n"
        "4. Envie o e-mail para CADA destinatário da lista usando send_email.\n\n"
        "Você NÃO lê arquivos de skill diretamente — use sempre consultar_skill para isso.\n"
        "Seja fiel ao formato da skill: use os emojis, campos e padrão de assunto indicados."
    ),
)

print("Orquestrador criado.")

---
## Parte D — Testes end-to-end

In [ ]:
# Célula 13 — Helper de invocação

def invocar_orquestrador(prompt: str) -> str:
    resultado = orquestrador.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 20},
    )
    return resultado["messages"][-1].content

print("Helper pronto.")

In [ ]:
# Célula 14 — Teste 1: alerta de falha
#
# Observe:
# - O orquestrador chama consultar_skill
# - O sub-agente lê cabeçalhos, decide e lê alerta_falha.md completo
# - O orquestrador usa as instruções para redigir e envia para todos

print("=== ALERTA DE FALHA ===")
resposta = invocar_orquestrador(
    "Envia um alerta de falha. O sistema de pagamentos está fora do ar desde as 14h."
)
print(resposta)

In [ ]:
# Célula 15 — Teste 2: alerta de produção atingida
#
# Desta vez o sub-agente deve escolher alerta_producao.md

print("=== META DE PRODUÇÃO ATINGIDA ===")
resposta = invocar_orquestrador(
    "Envia o alerta de produção. A linha 3 bateu a meta de 500 unidades neste turno."
)
print(resposta)

In [ ]:
# Célula 16 — Verificar e-mails recebidos
# Confirme que os e-mails chegaram na caixa de entrada

r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
inbox = r.json()
print(f"{inbox['count']} mensagem(ns) na caixa de entrada:\n")
for msg in inbox["messages"]:
    print(f"  [{msg['id']}] De: {msg['from']:<25} | {msg['subject']}")

---
## Resumo do Dia 5 e do Curso

| Conceito | O que aprendemos |
|---|---|
| Skill (`.md`) | Arquivo de instrução que ensina o agente a formatar uma mensagem |
| `read_skill(header_only)` | Scan rápido de cabeçalhos antes de ler o arquivo completo |
| Sub-agente | Agente especializado com escopo limitado de tools e responsabilidades |
| Tool wrapper | Encapsular um agente como tool para que outro agente possa invocá-lo |
| Orquestrador | Agente que coordena sub-agentes e executa as ações finais |
| Separação de responsabilidades | Sub-agente lê; orquestrador envia. Cada um faz uma coisa. |

---

## Arco completo do curso

| Dia | Conceito central |
|---|---|
| 1 | Conexão ao LLM + primeira tool + primeiro agente |
| 2 | Múltiplas tools + agente multi-tool + memória de sessão |
| 3 | RAG com FAISS — o agente consulta conhecimento externo |
| 4 | LangGraph — controle explícito de fluxo + human-in-the-loop |
| 5 | Orquestração — agentes delegando para agentes via skills |